# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested |
|---|---|---|
| v0.1.0 | Voyce | 13 Jul 2026 |
| v0.2.0 | Voyce | 6 Aug 2026 |


In [ ]:
# Import supported public objects from the root package.
# Make sure the Fabric environment already has FabricOps installed as a custom library.

from fabricops_kit import (
    DataAgreementConfig,
    FabricStore,
    FrameworkConfig,
    GovernanceConfig,
    PathConfig,
    setup_metadata_tables,
    setup_notebook,
)

## Path config

Define the environment and the logical Fabric stores available to downstream notebooks.
Logical store names are project-configurable. FabricOps reserves `metadata` for the metadata Lakehouse.


In [ ]:
# Change this if needed for your own custom-defined environments, for example: dev, qat, prd.
ENV = "dev"

ENV_PATHS = {
    ENV: {
        "source": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="ed3aad28-de5d-43d5-8a97-a6988901c921",
            name="Source",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "unified": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="107d4b73-7c0e-4ce1-8da8-7602ac4a1372",
            name="Unified",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "product": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="185346e5-6ce4-40c8-852a-185f1a933d20",
            name="Product",
            kind="warehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "metadata": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="329b3989-4546-4331-b9ff-df898d49ee73",
            name="Metadata",
            kind="lakehouse",
            schema_enabled=True,
            # Store-level default only. Canonical metadata writers route tables to
            # FrameworkConfig governance and engineering schemas by table ownership.
            schema="governance",
        ),
    }
}

PATH_CONFIG = PathConfig(paths=ENV_PATHS)

## 01_governance metadata intake config

The Steward and Agreement widgets in `01_governance` expose only lightweight business fields. Add organization-specific fields here; the widgets store those values in `custom_fields_json` without changing package code or table schemas.


In [ ]:
DATA_AGREEMENT_CONFIG = DataAgreementConfig(
    metadata_tables={
        "data_steward": "METADATA_DATA_STEWARD",
        "data_agreement": "METADATA_DATA_AGREEMENT",
    },
    steward_role_options=[
        "Data Owner",
        "Data Steward",
        "Data Custodian",
        "Governance Reviewer",
        "Business Approver",
    ],
    data_steward_widget={
        "visible_columns": [
            "steward_name", "steward_role", "contact", "effective_from", "effective_to",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
    data_agreement_widget={
        "visible_columns": [
            "agreement_name", "domain", "provider_steward_id", "recipient_steward_id",
            "recipient", "start_date", "expiry_date", "business_purpose",
        ],
        "approved_usage_options": ["internal cross domain", "internal single domain", "research", "external"],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
)

## 01_governance enrichment config

Configure information-classification labels and optional AI-assisted Description and Classification suggestions. AI suggestions remain editable and are never saved automatically.


In [ ]:
GOVERNANCE_CONFIG = GovernanceConfig(
    sensitivity_labels=["Public", "Internal", "Confidential", "Restricted"],
    ai_enrichment={
        "enabled": True,
        "description_prompt": """Write a concise business description for this table or column.

Use the supplied technical metadata, profile evidence, and context.
Describe what the data represents in business terms.
Do not invent unsupported business meaning.
Return only the proposed description.""",
        "classification_prompt": """Classify this data according to the organisation's information-classification policy.

Use only one of the configured classification labels.
Base the recommendation on the supplied table or column name, datatype, description, profile evidence, and available context.
Return only the proposed classification label.""",
        "sensitive_data_prompt": """Suggest advisory Sensitive Data Guardrails for active canonical Catalogue columns.

Use only the supplied metadata, Enrichment, and profile context; do not request or include raw values.
Use only Tokenize, Mask, Bucket, or Remove and return a JSON list with column, treatment, action, and explicit treatment parameters.
Treat Classification as one input signal only, keep proposed Bucket boundaries and labels explicit, and leave final approval to Governance.""",
    },
)

## Config compiler and bootstrap

Assemble the shared config once. Downstream functions resolve store paths and canonical metadata schemas from this config rather than duplicating those choices in notebook code.


In [ ]:
# FabricOps audit and technical timestamp timezone.
# UTC is the portable default. Use a valid IANA timezone when local audit time is required.
FABRICOPS_AUDIT_TIMEZONE = "Asia/Singapore"

CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    data_agreement_config=DATA_AGREEMENT_CONFIG,
    audit_timezone=FABRICOPS_AUDIT_TIMEZONE,
)

# Validate every logical store declared for this environment.
RUN_CONTEXT = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=list(CONFIG.path_config.paths[ENV]),
)

In [ ]:
# Expose the minimal shared context consumed by downstream FabricOps resolvers.
# Store identities and metadata schemas are resolved from CONFIG when needed.
import builtins

FABRIC_CONTEXT = {
    "env": ENV,
    "config": CONFIG,
    "runtime_metadata": RUN_CONTEXT.runtime_metadata,
}
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT

print(f"Active Fabric context initialized for environment: {ENV}")

In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")

for store_name, store in CONFIG.path_config.paths[ENV].items():
    print(f"- {store_name}: {store.name}")

## Metadata table setup

Create or validate the canonical FabricOps metadata tables in the configured `metadata` Lakehouse.
FabricOps resolves each table to its owning `governance` or `engineering` schema from `FrameworkConfig`; this notebook does not assign one schema to all metadata tables.


In [ ]:
METADATA_TABLE_SETUP = setup_metadata_tables(
    spark=spark,
    config=CONFIG,
    env=ENV,
    require_active_steward=False,
)

In [ ]:
print("FabricOps environment ready")
print(f"- audit timezone: {CONFIG.audit_timezone}")
print(f"- governance metadata schema: {CONFIG.governance_metadata_schema}")
print(f"- engineering metadata schema: {CONFIG.engineering_metadata_schema}")
print(f"- active metadata tables: {METADATA_TABLE_SETUP['active_metadata_table_count']}")